# Build, Shape, and Extend an OpenClaw Agent on AMD

---

**What you have in front of you:**
- This notebook — the workshop guide. Read it, run the occasional cell.
- A terminal on the side — where the real interaction happens via the OpenClaw TUI.

**Format:**
- ▶ **Run this cell** → execute it (Shift+Enter)
- 💬 **In the TUI** → go to your terminal and type / paste this
- 👀 **Behind the scenes** → we peek at what the agent actually wrote to disk

---
## Section 0 — Setup

Run these three cells once. They verify the model server, connect OpenClaw to it, and start the gateway.

In [ ]:
import urllib.request, json, subprocess, time, pathlib, sys

try:
    req = urllib.request.Request(
        "http://localhost:8090/v1/models",
        headers={"Authorization": "Bearer abc-123"}
    )
    with urllib.request.urlopen(req) as r:
        models = json.loads(r.read())
    print("✅ SGLang is running")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
except Exception as e:
    print(f"❌ SGLang not reachable: {e}")

In [ ]:
BASE_URL = "http://localhost:8090/v1"
API_KEY  = "abc-123"
MODEL_ID = "qwen3-5-122b"

subprocess.run(["openclaw", "config", "set", "gateway.mode", "local"], check=True)
subprocess.run(["openclaw", "config", "set", "agents.defaults.model", f"sglang/{MODEL_ID}"], check=True)

config_path = pathlib.Path.home() / ".openclaw" / "openclaw.json"
with config_path.open() as f:
    cfg = json.load(f)

cfg.setdefault("models", {}).setdefault("providers", {})["sglang"] = {
    "baseUrl": BASE_URL, "apiKey": API_KEY, "api": "openai-completions",
    "models": [{"id": MODEL_ID, "name": MODEL_ID, "reasoning": True,
                "input": ["text"], "cost": {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
                "contextWindow": 131072, "maxTokens": 8192}]
}
with config_path.open("w") as f:
    json.dump(cfg, f, indent=2)
print("✅ OpenClaw configured → Qwen3.5-122B on SGLang")

In [ ]:
log_file = pathlib.Path("/tmp/openclaw-gateway.log")
subprocess.run(["pkill", "-9", "-f", "openclaw-gateway"], capture_output=True)
time.sleep(1)
proc = subprocess.Popen(
    ["openclaw", "gateway", "run", "--bind", "loopback", "--port", "18789", "--force"],
    stdout=log_file.open("w"), stderr=subprocess.STDOUT,
)
time.sleep(4)
if proc.poll() is None:
    print(f"✅ Gateway running (PID {proc.pid})")
else:
    print("❌ Gateway failed:\n", log_file.read_text()[-600:])

---
## The Map

Before we start, here's everything you'll touch today and what it does:

| Concept | What it is | Where it lives |
|---|---|---|
| **Gateway** | Connects OpenClaw to the model running on AMD hardware | Process on port 18789 |
| **Workspace** | The agent's "brain" — files it reads on every message | `~/.openclaw/workspace/` |
| **Tools** | How the agent acts on the world: read files, run shell commands, write edits | Defined in `TOOLS.md` |
| **Skills** | Saved, reusable workflows the agent can follow on demand | `workspace/skills/` |

The key insight: **there is no magic**. The agent's memory, personality, and skills are plain markdown files on disk. You can read them, edit them, and version-control them. That's what this workshop demonstrates.

---
## Section 1 — Onboarding

### 💬 In the TUI — start OpenClaw

```bash
openclaw
```

OpenClaw will ask you a few questions — your name, how you like to work, what kind of tone you prefer. Answer however you like.

When you're done, come back here.

### 👀 Behind the scenes — what just happened

Those answers didn't just disappear. The agent wrote them to files in its **workspace** — a folder it reads on every message to remember who you are and how to behave.

Run the next two cells to see exactly what was written.

In [ ]:
workspace = pathlib.Path.home() / ".openclaw" / "workspace"

print("=" * 60)
print("IDENTITY.md")
print("=" * 60)
print((workspace / "IDENTITY.md").read_text())

In [ ]:
print("=" * 60)
print("SOUL.md")
print("=" * 60)
print((workspace / "SOUL.md").read_text())

The workspace has 8 files that get injected into every session:

```
~/.openclaw/workspace/
├── SOUL.md       ← WHO the agent is: values, personality, tone
├── AGENTS.md     ← HOW it operates: startup rules, memory, red lines
├── IDENTITY.md   ← WHAT others see: name, emoji, public-facing metadata
├── USER.md       ← WHO you are: context the agent reads about you
├── TOOLS.md      ← local environment notes (SSH hosts, device names, etc.)
├── MEMORY.md     ← long-term curated memory across sessions
├── HEARTBEAT.md  ← checklist for periodic background checks
├── BOOTSTRAP.md  ← first-run ritual (deleted after onboarding)
└── memory/       ← daily session logs (today + yesterday auto-loaded)
```

**SOUL.md is personality. AGENTS.md is policy.** Every message the agent receives, it re-reads both. The startup sequence is literally written into `AGENTS.md`:

> *Before doing anything else: read SOUL.md, read USER.md, read today's memory log.*

Run the next cell to see AGENTS.md:"

In [ ]:
print("=" * 60)
print("AGENTS.md  (truncated)")
print("=" * 60)
agents_text = (workspace / "AGENTS.md").read_text()
print(agents_text[:1200] + "\n\n... (truncated)")

### Prove it — layer a rule into AGENTS.md before the bug fix

`SOUL.md` is personality — don't touch that. `AGENTS.md` is policy — that's where we add a rule about *how* to behave when debugging.

The next cell appends one section to `AGENTS.md`. After this, every debug explanation from the agent will follow those rules. Run it and see the before/after.

In [ ]:
debug_rule = """
---

## Debugging Policy (added during workshop)

When investigating and explaining bugs:
- State the bug in two sentences max: file, line, what it does wrong
- No filler phrases — just the root cause and the fix
- Use bullet points, never paragraphs
- After fixing, report exactly: file changed, line changed, before, after

"""

agents_path = workspace / "AGENTS.md"
original = agents_path.read_text()

print("── AGENTS.md BEFORE (last 200 chars) ──────────────────────")
print("..." + original[-200:])

agents_path.write_text(original + debug_rule)

print("\n── APPENDED ────────────────────────────────────────────────")
print(debug_rule)

The agent that does the bug fix in Section 4 will read the full file — original soul plus this new section. Watch how it explains the root cause compared to a plain `openclaw chat` with no shaping.

This is the pattern for real use too: you don't replace the soul for every task, you layer specialised instructions on top of a stable base.

---
## Section 2 — Pull the App

Now let's give the agent something to work with. Ask it to clone the typing app and set it up.

### 💬 In the TUI

```
Clone https://github.com/Mahdi-CV/open_type_faster and install its dependencies
```

Watch it use shell tools to clone, inspect the requirements, and install. This is the **act** phase — the agent doesn't just answer, it does.

---
## Section 3 — Run the App and Spot the Bug

### ⌨️ Open a new terminal and run the app yourself

```bash
cd ~/.openclaw/workspace/open_type_faster && ./venv/bin/python main.py
```

Type for 30 seconds. When the results screen appears — look at the accuracy.

> **Something is off. What do you see?**

---

The tests make the bug concrete. Run this cell:

In [ ]:
repo = workspace / "open_type_faster"
python = repo / "venv" / "bin" / "python"

result = subprocess.run(
    [str(python), "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True, cwd=str(repo)
)
print(result.stdout[-3000:])

3 tests failing, all in `TestAccuracy`:

- `_accuracy(5, 10)` should return `50.0` — returns `0`
- `_accuracy(2, 3)` should return `66.7` — returns `0`
- A session with one error should show `80.0%` — shows `0`

The accuracy calculation always returns 0. Ask the agent to find out why.

---
## Section 4 — Ask the Agent to Fix It

### 💬 In the TUI

```
The typing app tests are failing — accuracy always returns 0. Run the tests, find the bug in the source, fix it, and re-run to verify.
```

Watch the loop the agent runs:

1. **Run** `pytest` — see which tests fail and what they expect
2. **Read** the failing test — understand the correct behaviour
3. **Read** `stats.py` — trace the calculation back to the bug
4. **Fix** the minimal change needed
5. **Re-run** to verify

This think → act → observe → repeat loop is what separates an agent from a chatbot.

In [ ]:
# Run after the agent says it's done
result = subprocess.run(
    [str(python), "-m", "pytest", "tests/", "-v"],
    capture_output=True, text=True, cwd=str(repo)
)
print(result.stdout[-2000:])
passed = "failed" not in result.stdout
print("\n" + ("✅ All tests passing" if passed else "❌ Some tests still failing"))

---
## Section 5 — Turn It Into a Skill

What the agent just did was a one-off. A **skill** packages that behaviour so it can be invoked on any Python project — not just this one. Skills live in `~/.openclaw/workspace/skills/` as a folder containing a `SKILL.md` file with YAML frontmatter and step-by-step instructions. OpenClaw injects them into the system prompt automatically.

### 💬 In the TUI

```
Create a skill called pytest-debugger in the skills directory. The SKILL.md must define these steps: 1) Read the tests/ folder to understand what is being tested. 2) Run pytest with verbose output and short tracebacks. 3) For each failing test, read the source file it references. 4) Identify the minimal fix — do not rewrite functions. 5) Apply the fix and re-run pytest to confirm. 6) Report: file changed, line changed, what was wrong, what the fix was.
```

When the agent is done, run the next cell to see what it wrote.

In [ ]:
# SKILL.md can be uppercase or lowercase depending on what the agent wrote
# Search for either
skill_dir = workspace / "skills" / "pytest-debugger"
skill_file = None
for candidate in ["SKILL.md", "skill.md"]:
    if (skill_dir / candidate).exists():
        skill_file = skill_dir / candidate
        break

if skill_file:
    print("=" * 60)
    print(f"skills/pytest-debugger/{skill_file.name}")
    print("=" * 60)
    print(skill_file.read_text())
else:
    print(f"Skill not found at {skill_dir}")
    print("Check what the agent created:")
    for p in workspace.rglob("*SKILL*"):
        print(f"  {p}")

That file is all it takes. The skill lives in the workspace alongside `IDENTITY.md` and `SOUL.md` — OpenClaw injects it into every session automatically. Any time you ask the agent to debug a Python project, it has these exact steps to follow.

Skills are just markdown files. You can read them, edit them, and version-control them.

---
## Section 6 — Use the Skill

The skill is now in the workspace. OpenClaw injects it into the agent's system prompt automatically — there's no special command to invoke it. You just describe what you want in natural language, and the agent uses the skill instructions as its guide.

### 💬 In the TUI

```
Use the pytest-debugger skill to find and fix the failing tests in ~/.openclaw/workspace/open_type_faster
```

The agent will read the `SKILL.md` steps and follow them — without any further guidance from you.

---
## Section 7 — ClawHub

Everything built so far is local to your workspace. ClawHub makes it shareable.

```
Local skill  →  openclaw skill publish  →  ClawHub registry
Anyone       →  openclaw skill install pytest-debugger  →  their workspace
```

The skill travels with its context — whoever installs it gets the same behaviour, not a blank agent.

---

## What Happened Here

| Step | What the agent did | Where it wrote it |
|---|---|---|
| Onboarding | Learned your name, style, preferences | `IDENTITY.md`, `SOUL.md` |
| Clone | Pulled a real repo and set it up | Shell tools |
| Debug | Ran tests, traced a bug, applied a fix | Edited `stats.py` directly |
| Skill | Packaged the debugging approach | `skills/pytest-debugger/skill.md` |

Every one of those outputs is a plain file you can read, edit, and share.

---

## Challenge — Pick a Track

| Track | What to do |
|---|---|
| **Bug Hunter** | Introduce your own bug, swap repos with a neighbour, fix each other's using the agent |
| **Skill Builder** | Extend `pytest-debugger` to write a `bug_report.md` after each fix |
| **Soul Shaper** | Edit `SOUL.md` to change how the agent explains things — re-run the debug task and compare |

*Built something interesting? Share it and you may qualify for additional AMD Developer Cloud credits.*